# ANARCI + BLASTP for TCR3d alpha/beta sequences

A clean, independent Google Colab workflow for **ANARCI + NCBI BLASTP only**. It accepts the new TCR3d FASTA bundle, extracts IMGT CDRs, performs within-chain all-vs-all BLASTP searches, and verifies a reported BLAST bit score by hand.

**Run the notebook from top to bottom. A CPU runtime is sufficient.**


## The workflow

1. Upload `TCR3d_FASTA_files_bundle.zip` (or a plain protein FASTA).
2. Validate and normalize the sequences.
3. Run **ANARCI only on TCR alpha/beta models** and number accepted variable domains with IMGT.
4. Extract CDR1, CDR2, and CDR3 by **IMGT position**, not by fixed string offsets.
5. Search alpha against alpha and beta against beta with the one search engine named in this notebook.
6. Export transparent tables, runtime measurements, alignments, and a manual bit-score worksheet.

### Keep the three score meanings separate

| Column | Produced by | What is being scored |
|---|---|---|
| `anarci_bitscore` | ANARCI/HMMER | Sequence-to-profile-HMM evidence for an antigen-receptor domain |
| `blast_bitscore` | BLASTP | A pairwise local alignment normalized to bits |

The old `bitscore` column beside your CDRs was the **ANARCI/HMMER bit score**. It was not a BLAST or MMseqs2 score. This notebook preserves the tool name in every score column to prevent that mix-up.


## 1. Install only the software used here

This notebook intentionally does **not** install or call MMseqs2.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y hmmer ncbi-blast+ time
%pip install -q "ANARCI==2026.2.13.2" "biopython==1.85" pandas numpy


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package time.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../time_1.9-0.1build2_amd64.deb ...
Unpacking time (1.9-0.1build2) ...
Selecting previously unselected package libdivsufsort3:amd64.
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package ncbi-data.
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64.deb 

In [ ]:
import subprocess
import sys
from importlib.metadata import version
print("ANARCI", version("ANARCI"))
print(subprocess.check_output(["blastp", "-version"], text=True).splitlines()[0])
print(sys.version)


ANARCI 2026.2.13.2
blastp: 2.12.0+
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 2. Settings


In [ ]:
from pathlib import Path
import os

# Input -----------------------------------------------------------------------
# When uploading the provided zip, this exact member is selected by default.
# Alternatives in that zip:
#   TCR3d_ClassI_TCR_AB.fasta
#   TCR3d_ClassII_TCR_AB.fasta
#   TCR3d_All_Complex_Chains.fasta  (ANARCI will reject non-TCR chains)
FASTA_MEMBER_NAME = "TCR3d_ClassI_ClassII_TCR_AB.fasta"

# Set to a Google Drive/local path instead of uploading, or leave as None.
INPUT_PATH = None

# ANARCI / search settings -----------------------------------------------------
THREADS = min(2, os.cpu_count() or 1)
ANARCI_BATCH_SIZE = 200
MIN_INPUT_LENGTH = 70
REMOVE_EXACT_VARIABLE_DOMAIN_DUPLICATES = True
MIN_PAIRWISE_COVERAGE = 0.70
MAX_TARGET_SEQS = 100000

WORK_DIR = Path("/content/anarci_blast_tcr3d")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "results"
SEARCH_DIR = WORK_DIR / "search"
for directory in (INPUT_DIR, OUTPUT_DIR, SEARCH_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Threads: {THREADS}")
print(f"Default archive member: {FASTA_MEMBER_NAME}")


Threads: 2
Default archive member: TCR3d_ClassI_ClassII_TCR_AB.fasta


## 3. Upload and select a FASTA


In [ ]:
import re
import shutil
import zipfile
from google.colab import files


def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe path in zip: {member.filename}")
        archive.extractall(destination)


def choose_fasta(path, requested_member):
    path = Path(path)
    if path.suffix.lower() == ".zip":
        extracted = INPUT_DIR / "extracted_bundle"
        extracted.mkdir(parents=True, exist_ok=True)
        safe_extract_zip(path, extracted)
        candidates = sorted(
            p for p in extracted.rglob("*")
            if p.is_file() and p.suffix.lower() in {".fa", ".faa", ".fasta"}
        )
        exact = [p for p in candidates if p.name == requested_member]
        if exact:
            return exact[0], candidates
        if len(candidates) == 1:
            return candidates[0], candidates
        names = "\n".join(f"  - {p.name}" for p in candidates)
        raise FileNotFoundError(
            f"Could not find {requested_member!r}. Set FASTA_MEMBER_NAME to one of:\n{names}"
        )
    if path.suffix.lower() not in {".fa", ".faa", ".fasta"}:
        raise ValueError("Input must be a .zip, .fa, .faa, or .fasta file")
    return path, [path]


if INPUT_PATH is None:
    print("Upload TCR3d_FASTA_files_bundle.zip or a plain protein FASTA:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded")
    uploaded_name = next(iter(uploaded))
    uploaded_path = INPUT_DIR / Path(uploaded_name).name
    shutil.copy2(uploaded_name, uploaded_path)
else:
    uploaded_path = Path(INPUT_PATH)
    if not uploaded_path.exists():
        raise FileNotFoundError(uploaded_path)

FASTA_PATH, fasta_candidates = choose_fasta(uploaded_path, FASTA_MEMBER_NAME)
print(f"Selected FASTA: {FASTA_PATH}")
print("FASTA files available from this input:")
for candidate in fasta_candidates:
    print(" -", candidate.name)


Upload TCR3d_FASTA_files_bundle.zip or a plain protein FASTA:


Saving TCR3d_FASTA_files_bundle.zip to TCR3d_FASTA_files_bundle.zip
Selected FASTA: /content/anarci_blast_tcr3d/input/extracted_bundle/TCR3d_ClassI_ClassII_TCR_AB.fasta
FASTA files available from this input:
 - TCR3d_All_Complex_Chains.fasta
 - TCR3d_ClassII_TCR_AB.fasta
 - TCR3d_ClassI_ClassII_TCR_AB.fasta
 - TCR3d_ClassI_TCR_AB.fasta


## 4. Parse and validate the protein sequences


In [ ]:
import pandas as pd
from Bio import SeqIO
from IPython.display import display

VALID_AA = set("ACDEFGHIKLMNPQRSTVWYXBZJUO")
AMBIGUOUS_TO_X = str.maketrans({aa: "X" for aa in "BZJUO"})


def expected_chain_from_header(header):
    match = re.search(r"(?:^|\|)expected_chain=([AB])(?:\||$)", header)
    if match:
        return match.group(1)
    match = re.search(r"(?:^|\|)tcr_type=(Alpha|Beta)(?:\||$)", header, re.I)
    if match:
        return {"alpha": "A", "beta": "B"}[match.group(1).lower()]
    return None


sequence_rows = []
rejected_rows = []
seen_ids = {}
for record_number, record in enumerate(SeqIO.parse(str(FASTA_PATH), "fasta"), start=1):
    header = record.description.strip()
    original_id = record.id
    seen_ids[original_id] = seen_ids.get(original_id, 0) + 1
    source_id = original_id if seen_ids[original_id] == 1 else f"{original_id}__copy{seen_ids[original_id]}"
    raw_sequence = str(record.seq).upper().replace("*", "")
    invalid = sorted(set(raw_sequence) - VALID_AA)
    normalized = raw_sequence.translate(AMBIGUOUS_TO_X)
    reason = None
    if invalid:
        reason = f"invalid characters: {''.join(invalid)}"
    elif len(normalized) < MIN_INPUT_LENGTH:
        reason = f"shorter than {MIN_INPUT_LENGTH} aa"
    if reason:
        rejected_rows.append({
            "record_number": record_number,
            "source_id": source_id,
            "length": len(normalized),
            "reason": reason,
        })
        continue
    sequence_rows.append({
        "record_number": record_number,
        "source_id": source_id,
        "original_id": original_id,
        "header": header,
        "expected_chain": expected_chain_from_header(header),
        "sequence": normalized,
        "input_length": len(normalized),
    })

sequences_df = pd.DataFrame(sequence_rows)
input_rejected_df = pd.DataFrame(rejected_rows)
if sequences_df.empty:
    raise RuntimeError("No valid protein sequences remained after input validation")

sequences_df.to_csv(OUTPUT_DIR / "input_sequence_manifest.csv", index=False)
input_rejected_df.to_csv(OUTPUT_DIR / "input_rejected.csv", index=False)
print(f"Accepted input records: {len(sequences_df):,}")
print(f"Rejected before ANARCI: {len(input_rejected_df):,}")
display(sequences_df[["source_id", "expected_chain", "input_length"]].head())


Accepted input records: 1,189
Rejected before ANARCI: 5


,source_id,expected_chain,input_length
0,1AO7|mhc_class=I|chain=D|expected_chain=A|tcr_...,A,204
1,1AO7|mhc_class=I|chain=E|expected_chain=B|tcr_...,B,245
2,1BD2|mhc_class=I|chain=D|expected_chain=A|tcr_...,A,204
3,1BD2|mhc_class=I|chain=E|expected_chain=B|tcr_...,B,244
4,1FO0|mhc_class=I|chain=A|expected_chain=A|tcr_...,A,116


## ANARCI and the IMGT CDR boundaries

ANARCI compares each sequence to antigen-receptor **profile HMMs**. Here `allow={"A", "B"}` means only TCR alpha and beta models are accepted. A significant model match identifies the variable domain, classifies the chain, and assigns IMGT position labels.

CDRs are then selected by those labels:

| Region | IMGT positions used here |
|---|---:|
| CDR1 | 27–38 |
| CDR2 | 56–65 |
| CDR3 | 105–117 |

This is why CDRs should **not** be sliced from fixed Python indices. Insertions and variable framework lengths can move the same IMGT position to different raw-sequence offsets. IMGT insertion codes are retained in the `*_numbered` columns.

ANARCI documentation: [PyPI package and output definition](https://pypi.org/project/ANARCI/); method paper: [Dunbar & Deane, 2016](https://pmc.ncbi.nlm.nih.gov/articles/PMC4708101/).


In [ ]:
import time
from anarci import anarci as run_anarci

sequence_pairs = list(zip(sequences_df["source_id"], sequences_df["sequence"]))
numbered_results = []
detail_results = []
hit_tables = []

anarci_start = time.perf_counter()
for start in range(0, len(sequence_pairs), ANARCI_BATCH_SIZE):
    batch = sequence_pairs[start:start + ANARCI_BATCH_SIZE]
    print(f"ANARCI records {start + 1:,}–{start + len(batch):,} of {len(sequence_pairs):,}")
    numbered, details, hits = run_anarci(
        batch,
        scheme="imgt",
        allow={"A", "B"},
        output=False,
        ncpu=THREADS,
    )
    numbered_results.extend(numbered)
    detail_results.extend(details)
    hit_tables.extend(hits)
anarci_seconds = time.perf_counter() - anarci_start
print(f"ANARCI finished in {anarci_seconds:.2f} seconds")


ANARCI records 1–200 of 1,189
ANARCI records 201–400 of 1,189
ANARCI records 401–600 of 1,189
ANARCI records 601–800 of 1,189
ANARCI records 801–1,000 of 1,189
ANARCI records 1,001–1,189 of 1,189
ANARCI finished in 26.68 seconds


In [ ]:
REGIONS = {
    "FR1_IMGT": (1, 26),
    "CDR1_IMGT": (27, 38),
    "FR2_IMGT": (39, 55),
    "CDR2_IMGT": (56, 65),
    "FR3_IMGT": (66, 104),
    "CDR3_IMGT": (105, 117),
    "FR4_IMGT": (118, 128),
}


def occupied_items(numbered_domain, start=None, end=None):
    items = []
    for (position, insertion), aa in numbered_domain:
        position = int(position)
        if aa == "-":
            continue
        if start is not None and position < start:
            continue
        if end is not None and position > end:
            continue
        items.append((position, str(insertion).strip(), aa))
    return items


def item_sequence(items):
    return "".join(aa for _, _, aa in items)


def numbered_string(items):
    return ";".join(f"{position}{insertion}:{aa}" for position, insertion, aa in items)


metadata_by_id = sequences_df.set_index("source_id").to_dict("index")
anarci_rows = []
anarci_failure_rows = []

for index, source_id in enumerate(sequences_df["source_id"]):
    numbered_domains = numbered_results[index] or []
    domain_details = detail_results[index] or []
    if not numbered_domains:
        anarci_failure_rows.append({"source_id": source_id, "reason": "no significant A/B HMM hit"})
        continue
    for domain_index, domain_result in enumerate(numbered_domains):
        numbered_domain, domain_start, domain_end = domain_result
        info = domain_details[domain_index] if domain_index < len(domain_details) else {}
        chain_type = info.get("chain_type")
        if chain_type not in {"A", "B"}:
            anarci_failure_rows.append({
                "source_id": source_id,
                "reason": f"unexpected ANARCI chain type: {chain_type}",
            })
            continue
        meta = metadata_by_id[source_id]
        full_items = occupied_items(numbered_domain)
        row = {
            "source_id": source_id,
            "original_id": meta["original_id"],
            "header": meta["header"],
            "input_length": meta["input_length"],
            "expected_chain": meta["expected_chain"],
            "chain_type": chain_type,
            "chain_label_match": (
                None if meta["expected_chain"] is None
                else meta["expected_chain"] == chain_type
            ),
            "domain_number": domain_index + 1,
            "input_domain_start_0based": domain_start,
            "input_domain_end_0based": domain_end,
            "species_model": info.get("species"),
            "anarci_bitscore": info.get("bitscore", info.get("score")),
            "anarci_evalue": info.get("evalue"),
            "variable_domain_sequence": item_sequence(full_items),
            "variable_domain_numbered": numbered_string(full_items),
        }
        for region, (region_start, region_end) in REGIONS.items():
            items = occupied_items(numbered_domain, region_start, region_end)
            row[region] = item_sequence(items)
            row[f"{region}_length"] = len(items)
            row[f"{region}_numbered"] = numbered_string(items)
        anarci_rows.append(row)

anarci_df = pd.DataFrame(anarci_rows)
anarci_failures_df = pd.DataFrame(anarci_failure_rows)
if anarci_df.empty:
    raise RuntimeError("ANARCI found no TCR alpha/beta variable domains")

anarci_df.to_csv(OUTPUT_DIR / "ANARCI_CDR_IMGT_results.csv", index=False)
anarci_failures_df.to_csv(OUTPUT_DIR / "ANARCI_failures.csv", index=False)
chain_mismatches_df = anarci_df.loc[anarci_df["chain_label_match"] == False].copy()
chain_mismatches_df.to_csv(OUTPUT_DIR / "ANARCI_chain_label_mismatches.csv", index=False)

print(f"Numbered A/B domains: {len(anarci_df):,}")
print(f"ANARCI failures: {len(anarci_failures_df):,}")
print(f"Expected-vs-predicted chain mismatches: {len(chain_mismatches_df):,}")
display(anarci_df[[
    "source_id", "chain_type", "anarci_bitscore",
    "CDR1_IMGT", "CDR2_IMGT", "CDR3_IMGT"
]].head(10))


Numbered A/B domains: 1,164
ANARCI failures: 33
Expected-vs-predicted chain mismatches: 21


,source_id,chain_type,anarci_bitscore,CDR1_IMGT,CDR2_IMGT,CDR3_IMGT
0,1AO7|mhc_class=I|chain=D|expected_chain=A|tcr_...,A,129.9,DRGSQS,IYSNGD,AVTTDSWGKLQ
1,1AO7|mhc_class=I|chain=E|expected_chain=B|tcr_...,B,143.1,MNHEY,SVGAGI,ASRPGLAGGRPEQY
2,1BD2|mhc_class=I|chain=E|expected_chain=B|tcr_...,B,148.9,MNHEY,SVGAGI,ASSYPGGGFYEQY
3,1FO0|mhc_class=I|chain=B|expected_chain=B|tcr_...,B,106.3,NSQYPW,LRSPGD,TCSADRVGNTLY
4,1G6R|mhc_class=I|chain=A|expected_chain=A|tcr_...,A,135.6,YSATPY,YYSGDPVV,AVSGFASALT
5,1G6R|mhc_class=I|chain=C|expected_chain=A|tcr_...,A,135.6,YSATPY,YYSGDPVV,AVSGFASALT
6,1G6R|mhc_class=I|chain=B|expected_chain=B|tcr_...,B,135.1,NNHNN,SYGAGS,ASGGGGTLY
7,1G6R|mhc_class=I|chain=D|expected_chain=B|tcr_...,B,135.1,NNHNN,SYGAGS,ASGGGGTLY
8,1KJ2|mhc_class=I|chain=A|expected_chain=A|tcr_...,A,147.8,DSTFNY,IRSVSDK,AARYQGGRALI
9,1KJ2|mhc_class=I|chain=D|expected_chain=A|tcr_...,A,147.8,DSTFNY,IRSVSDK,AARYQGGRALI


## Prepare alpha and beta variable-domain FASTAs

Alpha and beta are searched separately because a high alpha-to-beta similarity is not the biological comparison of interest. Exact duplicate variable domains can be represented once to avoid measuring the same PDB-derived sequence repeatedly; the manifest maps every representative back to all source records.


In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def make_search_manifest(numbered_df, deduplicate=True):
    work = numbered_df.copy()
    work["source_domain_id"] = work["source_id"].astype(str) + "__d" + work["domain_number"].astype(str)
    if deduplicate:
        grouped = (
            work.groupby(["chain_type", "variable_domain_sequence"], sort=False, dropna=False)
            .agg(
                source_count=("source_domain_id", "size"),
                source_domain_ids=("source_domain_id", lambda x: ";".join(map(str, x))),
                source_ids=("source_id", lambda x: ";".join(map(str, x))),
            )
            .reset_index()
        )
    else:
        grouped = work[["chain_type", "variable_domain_sequence", "source_domain_id", "source_id"]].copy()
        grouped["source_count"] = 1
        grouped["source_domain_ids"] = grouped["source_domain_id"]
        grouped["source_ids"] = grouped["source_id"]
        grouped = grouped.drop(columns="source_domain_id")
    counters = {"A": 0, "B": 0}
    search_ids = []
    for chain in grouped["chain_type"]:
        counters[chain] += 1
        search_ids.append(f"{chain}{counters[chain]:06d}")
    grouped.insert(0, "search_id", search_ids)
    return grouped


search_manifest_df = make_search_manifest(
    anarci_df,
    deduplicate=REMOVE_EXACT_VARIABLE_DOMAIN_DUPLICATES,
)
search_manifest_df.to_csv(OUTPUT_DIR / "pairwise_search_sequence_manifest.csv", index=False)

search_fasta_paths = {}
for chain in ("A", "B"):
    chain_df = search_manifest_df.loc[search_manifest_df["chain_type"] == chain]
    fasta_path = SEARCH_DIR / f"TCR_{chain}_variable_domains.fasta"
    records = [
        SeqRecord(Seq(row.variable_domain_sequence), id=row.search_id, description="")
        for row in chain_df.itertuples(index=False)
    ]
    SeqIO.write(records, fasta_path, "fasta")
    SeqIO.write(records, OUTPUT_DIR / fasta_path.name, "fasta")
    search_fasta_paths[chain] = fasta_path
    print(f"Chain {chain}: {len(chain_df):,} search sequences -> {fasta_path.name}")

print("Exact duplicate variable domains are represented once in the search; source IDs remain in the manifest.")


Chain A: 230 search sequences -> TCR_A_variable_domains.fasta
Chain B: 239 search sequences -> TCR_B_variable_domains.fasta
Exact duplicate variable domains are represented once in the search; source IDs remain in the manifest.


## NCBI BLASTP all-vs-all search

Scoring is deliberately transparent: BLOSUM62, gap open 11, gap extension 1, no SEG masking, and no composition-based score adjustment. This controlled setting makes the displayed alignment's raw score manually reproducible. It is an educational/reproducible configuration, not necessarily the default configuration of every BLAST web search.

NCBI defines the raw score as substitution scores plus affine gap costs and converts raw score \(S\) to bits with

\[
S' = \frac{\lambda S - \ln K}{\ln 2}.
\]

Source: [NCBI BLAST command-line manual](https://www.ncbi.nlm.nih.gov/books/NBK279684/).


In [ ]:
import subprocess
import time

BLAST_FIELDS = [
    "qseqid", "sseqid", "pident", "length", "nident", "mismatch", "gapopen",
    "qlen", "slen", "qstart", "qend", "sstart", "send", "evalue",
    "blast_bitscore", "blast_raw_score", "qseq", "sseq",
]
blast_frames = []
runtime_rows = []

for chain in ("A", "B"):
    fasta_path = search_fasta_paths[chain]
    db_prefix = SEARCH_DIR / f"blast_db_{chain}"
    output_path = SEARCH_DIR / f"blast_{chain}_all_vs_all.tsv"

    db_start = time.perf_counter()
    subprocess.run([
        "makeblastdb", "-in", str(fasta_path), "-dbtype", "prot",
        "-out", str(db_prefix), "-parse_seqids",
    ], check=True)
    db_seconds = time.perf_counter() - db_start

    command = [
        "blastp", "-query", str(fasta_path), "-db", str(db_prefix),
        "-out", str(output_path),
        "-outfmt", "6 " + " ".join([
            "qseqid", "sseqid", "pident", "length", "nident", "mismatch", "gapopen",
            "qlen", "slen", "qstart", "qend", "sstart", "send", "evalue",
            "bitscore", "score", "qseq", "sseq",
        ]),
        "-matrix", "BLOSUM62", "-gapopen", "11", "-gapextend", "1",
        "-comp_based_stats", "F", "-seg", "no", "-soft_masking", "false",
        "-max_hsps", "1", "-max_target_seqs", str(MAX_TARGET_SEQS),
        "-evalue", "10", "-num_threads", str(THREADS),
    ]
    search_start = time.perf_counter()
    subprocess.run(command, check=True)
    search_seconds = time.perf_counter() - search_start

    if output_path.stat().st_size:
        frame = pd.read_csv(output_path, sep="\t", names=BLAST_FIELDS)
        frame.insert(0, "chain_type", chain)
        blast_frames.append(frame)
    runtime_rows.append({
        "chain_type": chain,
        "sequence_count": int((search_manifest_df["chain_type"] == chain).sum()),
        "database_seconds": db_seconds,
        "search_seconds": search_seconds,
        "total_seconds": db_seconds + search_seconds,
    })

blast_all_df = pd.concat(blast_frames, ignore_index=True) if blast_frames else pd.DataFrame(columns=["chain_type"] + BLAST_FIELDS)
blast_runtime_df = pd.DataFrame(runtime_rows)
blast_runtime_df.to_csv(OUTPUT_DIR / "BLAST_runtime.csv", index=False)
display(blast_runtime_df)


,chain_type,sequence_count,database_seconds,search_seconds,total_seconds
0,A,230,0.089670,4.560522,4.650193
1,B,239,0.088895,3.985466,4.074362


In [ ]:
# Add coverage, remove self-hits, and keep one row per unordered reciprocal pair.
blast_all_df["qcov"] = (blast_all_df["qend"] - blast_all_df["qstart"] + 1) / blast_all_df["qlen"]
blast_all_df["tcov"] = (blast_all_df["send"] - blast_all_df["sstart"] + 1) / blast_all_df["slen"]
blast_nonself_df = blast_all_df.loc[blast_all_df["qseqid"] != blast_all_df["sseqid"]].copy()
blast_filtered_df = blast_nonself_df.loc[
    blast_nonself_df[["qcov", "tcov"]].min(axis=1) >= MIN_PAIRWISE_COVERAGE
].copy()
blast_filtered_df["pair_id"] = blast_filtered_df.apply(
    lambda row: "::".join(sorted((row["qseqid"], row["sseqid"]))), axis=1
)
blast_pairs_df = (
    blast_filtered_df.sort_values(["blast_bitscore", "pident"], ascending=False)
    .drop_duplicates(["chain_type", "pair_id"])
    .reset_index(drop=True)
)

blast_all_df.to_csv(OUTPUT_DIR / "BLAST_all_directed_hits.csv", index=False)
blast_filtered_df.to_csv(OUTPUT_DIR / "BLAST_coverage_filtered_directed_hits.csv", index=False)
blast_pairs_df.to_csv(OUTPUT_DIR / "BLAST_unique_pairs.csv", index=False)
print(f"All directed hits: {len(blast_all_df):,}")
print(f"Non-self directed hits at coverage >= {MIN_PAIRWISE_COVERAGE:.0%}: {len(blast_filtered_df):,}")
print(f"Unique unordered pairs: {len(blast_pairs_df):,}")
display(blast_pairs_df.head(10))

search_run_metadata = {
    "search_tool": "NCBI BLASTP",
    "blast_database_seconds": float(blast_runtime_df["database_seconds"].sum()),
    "blast_search_seconds": float(blast_runtime_df["search_seconds"].sum()),
    "blast_total_seconds": float(blast_runtime_df["total_seconds"].sum()),
    "pairwise_hits_after_coverage_filter": int(len(blast_filtered_df)),
    "unique_unordered_pairs": int(len(blast_pairs_df)),
}
RESULT_ARCHIVE_STEM = "ANARCI_BLAST_TCR3d_results"


All directed hits: 109,959
Non-self directed hits at coverage >= 70%: 109,366
Unique unordered pairs: 54,683


,chain_type,qseqid,sseqid,pident,length,nident,mismatch,gapopen,qlen,slen,...,sstart,send,evalue,blast_bitscore,blast_raw_score,qseq,sseq,qcov,tcov,pair_id
0,B,B000040,B000041,100.000,115,115,0,0,116,115,...,1,115,4.330000e-86,238.0,608,AGVIQSPRHEVTEMGQQVTLRCKPISGHDYLFWYRQTMMRGLELLI...,AGVIQSPRHEVTEMGQQVTLRCKPISGHDYLFWYRQTMMRGLELLI...,0.991379,1.000000,B000040::B000041
1,B,B000176,B000180,100.000,117,117,0,0,118,118,...,1,117,1.050000e-85,238.0,606,GAVVSQHPSWVISKSGTSVKIECRSLDFQATTMFWYRQFPKQSLML...,GAVVSQHPSWVISKSGTSVKIECRSLDFQATTMFWYRQFPKQSLML...,0.991525,0.991525,B000176::B000180
2,A,A000201,A000206,99.123,114,113,1,0,114,114,...,1,114,7.390000e-86,238.0,606,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000201::A000206
3,B,B000123,B000124,98.261,115,113,2,0,115,115,...,1,115,5.900000e-86,238.0,607,NAGVTQTPKFRVLKTGQSMTLLCAQDMNHEYMYWYRQDPGMGLRLI...,NAGVTQTPKFRVLKTGQSMTLLCAQDMNHEYMYWYRQDPGMGLRLI...,1.000000,1.000000,B000123::B000124
4,B,B000001,B000010,100.000,113,113,0,0,115,113,...,1,113,1.590000e-85,237.0,604,GVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLIHY...,GVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLIHY...,0.982609,1.000000,B000001::B000010
5,A,A000204,A000205,99.123,114,113,1,0,114,114,...,1,114,1.490000e-85,237.0,604,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000204::A000205
6,A,A000201,A000204,99.123,114,113,1,0,114,114,...,1,114,3.010000e-85,236.0,602,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000201::A000204
7,A,A000201,A000205,99.123,114,113,1,0,114,114,...,1,114,4.280000e-85,236.0,601,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000201::A000205
8,A,A000204,A000206,99.123,114,113,1,0,114,114,...,1,114,2.120000e-85,236.0,603,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000204::A000206
9,A,A000205,A000206,99.123,114,113,1,0,114,114,...,1,114,3.010000e-85,236.0,602,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,MQQVRQSPQSLTVWEGETAILNCSYENSAFDYFPWYQQFPGEGPAL...,1.000000,1.000000,A000205::A000206


## Manual BLAST bit-score calculation

First calculate the **raw alignment score** column by column:

- residue/residue: look up the BLOSUM62 value;
- first column of a gap: \(-(11+1)=-12\);
- each additional column in that same gap: \(-1\).

For BLOSUM62 with gap costs 11/1, the gapped statistical parameters used in this controlled BLASTP scoring system are \(\lambda=0.267\) and \(K=0.041\). BLAST reports bit scores to limited precision, so tiny rounding differences are expected.


In [ ]:
import math
from Bio.Align import substitution_matrices

BLOSUM62 = substitution_matrices.load("BLOSUM62")


def manual_raw_alignment_score(query_alignment, target_alignment, gap_open=11, gap_extend=1):
    # Score a displayed pairwise alignment column by column. This worksheet
    # assigns -(gap_open + k*gap_extend) to a gap of length k, so the first gap
    # column costs -(gap_open + gap_extend) and continuations cost -gap_extend.
    query_alignment = query_alignment.upper()
    target_alignment = target_alignment.upper()
    if len(query_alignment) != len(target_alignment):
        raise ValueError("The two aligned strings must have equal length")
    rows = []
    gap_state = None
    total = 0.0
    for column, (query_aa, target_aa) in enumerate(zip(query_alignment, target_alignment), start=1):
        if query_aa == "-" and target_aa == "-":
            raise ValueError(f"Both sequences have a gap in column {column}")
        if query_aa == "-" or target_aa == "-":
            current_gap = "query" if query_aa == "-" else "target"
            if current_gap == gap_state:
                contribution = -gap_extend
                reason = "gap extension"
            else:
                contribution = -(gap_open + gap_extend)
                reason = "gap open + first extension"
            gap_state = current_gap
        else:
            contribution = float(BLOSUM62[query_aa, target_aa])
            reason = f"BLOSUM62({query_aa},{target_aa})"
            gap_state = None
        total += contribution
        rows.append({
            "column": column,
            "query": query_aa,
            "target": target_aa,
            "reason": reason,
            "contribution": contribution,
            "running_raw_score": total,
        })
    return total, pd.DataFrame(rows)


def raw_to_bits(raw_score, lambda_value, K_value):
    return (lambda_value * raw_score - math.log(K_value)) / math.log(2)


TOY_QUERY = "ACDEFGHIK"
TOY_TARGET = "ACNEF-HIK"
toy_raw, toy_table = manual_raw_alignment_score(TOY_QUERY, TOY_TARGET)
display(toy_table)
print(f"Toy raw score = {toy_raw:g}")


BLAST_LAMBDA = 0.267
BLAST_K = 0.041
toy_bits = raw_to_bits(toy_raw, BLAST_LAMBDA, BLAST_K)
print(f"Toy bit score = ({BLAST_LAMBDA}*{toy_raw:g} - ln({BLAST_K})) / ln(2) = {toy_bits:.3f} bits")


,column,query,target,reason,contribution,running_raw_score
0,1,A,A,"BLOSUM62(A,A)",4.0,4.0
1,2,C,C,"BLOSUM62(C,C)",9.0,13.0
2,3,D,N,"BLOSUM62(D,N)",1.0,14.0
3,4,E,E,"BLOSUM62(E,E)",5.0,19.0
4,5,F,F,"BLOSUM62(F,F)",6.0,25.0
5,6,G,-,gap open + first extension,-12.0,13.0
6,7,H,H,"BLOSUM62(H,H)",8.0,21.0
7,8,I,I,"BLOSUM62(I,I)",4.0,25.0
8,9,K,K,"BLOSUM62(K,K)",5.0,30.0


Toy raw score = 30
Toy bit score = (0.267*30 - ln(0.041)) / ln(2) = 16.164 bits


### Verify a real BLAST hit from this run

The cell below selects the strongest non-self hit, recomputes its raw score from `qseq` and `sseq`, then normalizes it to bits. `raw_difference = 0` is the strongest check that the column-by-column arithmetic matches BLAST's score.


In [ ]:
if blast_nonself_df.empty:
    print("No non-self BLAST hit was available for this check.")
else:
    example_hit = blast_nonself_df.sort_values("blast_bitscore", ascending=False).iloc[0]
    manual_raw, real_contributions = manual_raw_alignment_score(example_hit.qseq, example_hit.sseq)
    manual_bits = raw_to_bits(manual_raw, BLAST_LAMBDA, BLAST_K)
    print(f"Pair: {example_hit.qseqid} vs {example_hit.sseqid}")
    print(f"BLAST raw score: {example_hit.blast_raw_score}")
    print(f"Manual raw score: {manual_raw:g}")
    print(f"Raw difference: {manual_raw - example_hit.blast_raw_score:g}")
    print(f"BLAST bit score: {example_hit.blast_bitscore}")
    print(f"Manual bit score: {manual_bits:.3f}")
    display(real_contributions)


Pair: B000180 vs B000176
BLAST raw score: 606
Manual raw score: 606
Raw difference: 0
BLAST bit score: 238.0
Manual bit score: 238.039


,column,query,target,reason,contribution,running_raw_score
0,1,G,G,"BLOSUM62(G,G)",6.0,6.0
1,2,A,A,"BLOSUM62(A,A)",4.0,10.0
2,3,V,V,"BLOSUM62(V,V)",4.0,14.0
3,4,V,V,"BLOSUM62(V,V)",4.0,18.0
4,5,S,S,"BLOSUM62(S,S)",4.0,22.0
...,...,...,...,...,...,...
112,113,T,T,"BLOSUM62(T,T)",5.0,588.0
113,114,R,R,"BLOSUM62(R,R)",5.0,593.0
114,115,L,L,"BLOSUM62(L,L)",4.0,597.0
115,116,T,T,"BLOSUM62(T,T)",5.0,602.0


### Your turn

Copy two **already aligned** strings (including `-` gap characters) into the cell. Before running it, calculate each contribution, the raw sum, and the normalized bit score on paper. Then compare.


In [ ]:
MY_QUERY_ALIGNMENT  = "ACDEFGHIK"
MY_TARGET_ALIGNMENT = "ACNEF-HIK"

my_raw, my_work = manual_raw_alignment_score(MY_QUERY_ALIGNMENT, MY_TARGET_ALIGNMENT)
my_bits = raw_to_bits(my_raw, BLAST_LAMBDA, BLAST_K)
display(my_work)
print(f"My raw score: {my_raw:g}")
print(f"My BLAST bit score: {my_bits:.3f}")


,column,query,target,reason,contribution,running_raw_score
0,1,A,A,"BLOSUM62(A,A)",4.0,4.0
1,2,C,C,"BLOSUM62(C,C)",9.0,13.0
2,3,D,N,"BLOSUM62(D,N)",1.0,14.0
3,4,E,E,"BLOSUM62(E,E)",5.0,19.0
4,5,F,F,"BLOSUM62(F,F)",6.0,25.0
5,6,G,-,gap open + first extension,-12.0,13.0
6,7,H,H,"BLOSUM62(H,H)",8.0,21.0
7,8,I,I,"BLOSUM62(I,I)",4.0,25.0
8,9,K,K,"BLOSUM62(K,K)",5.0,30.0


My raw score: 30
My BLAST bit score: 16.164


## What the ANARCI bit score means

ANARCI reports the bit score of the most significant **profile-HMM** hit. Conceptually, a profile-HMM score adds log-odds evidence from residue emissions and state transitions:

\[
S_{bits} \approx \sum_i \log_2 \left(\frac{P(\text{component}_i\mid\text{TCR profile})}{P(\text{component}_i\mid\text{null})}\right)
\]

A manual exact reconstruction of a real ANARCI score requires ANARCI's HMM, the Viterbi state path, and HMMER's null/bias corrections. Therefore the small exercise below demonstrates the mathematics but does **not** claim to reproduce ANARCI's full internal score. The ANARCI result table calls the column `anarci_bitscore` to make this distinction explicit.


In [ ]:
# Editable profile-HMM log-odds exercise.
toy_hmm_components = pd.DataFrame([
    {"component": "emit A at state 1", "profile_probability": 0.80, "null_probability": 0.10},
    {"component": "transition 1 -> 2", "profile_probability": 0.90, "null_probability": 0.50},
    {"component": "emit C at state 2", "profile_probability": 0.60, "null_probability": 0.05},
    {"component": "transition 2 -> 3", "profile_probability": 0.90, "null_probability": 0.50},
    {"component": "emit D at state 3", "profile_probability": 0.70, "null_probability": 0.10},
])
toy_hmm_components["log2_odds_bits"] = toy_hmm_components.apply(
    lambda row: math.log2(row["profile_probability"] / row["null_probability"]), axis=1
)
display(toy_hmm_components)
print(f"Toy profile-HMM score = {toy_hmm_components['log2_odds_bits'].sum():.3f} bits")
print("Try changing a profile_probability or null_probability, then rerun this cell.")


,component,profile_probability,null_probability,log2_odds_bits
0,emit A at state 1,0.8,0.10,3.000000
1,transition 1 -> 2,0.9,0.50,0.847997
2,emit C at state 2,0.6,0.05,3.584963
3,transition 2 -> 3,0.9,0.50,0.847997
4,emit D at state 3,0.7,0.10,2.807355


Toy profile-HMM score = 11.088 bits
Try changing a profile_probability or null_probability, then rerun this cell.


## Export and download

The zip contains ANARCI/CDR results, failures, chain-label mismatches, the search manifest, raw and filtered BLAST tables, runtime, and run metadata.


In [ ]:
import json
import platform
import sys

run_metadata = {
    "selected_fasta": FASTA_PATH.name,
    "input_records_accepted": int(len(sequences_df)),
    "input_records_rejected": int(len(input_rejected_df)),
    "anarci_domains": int(len(anarci_df)),
    "anarci_failures": int(len(anarci_failures_df)),
    "anarci_seconds": float(anarci_seconds),
    "threads": int(THREADS),
    "python": sys.version,
    "platform": platform.platform(),
}
run_metadata.update(search_run_metadata)
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2))

archive_base = WORK_DIR / RESULT_ARCHIVE_STEM
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1024:.1f} KiB)")
print("Output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {path.name}")
files.download(str(archive_path))


Created /content/anarci_blast_tcr3d/ANARCI_BLAST_TCR3d_results.zip (13041.0 KiB)
Output files:
 - ANARCI_CDR_IMGT_results.csv
 - ANARCI_chain_label_mismatches.csv
 - ANARCI_failures.csv
 - BLAST_all_directed_hits.csv
 - BLAST_coverage_filtered_directed_hits.csv
 - BLAST_runtime.csv
 - BLAST_unique_pairs.csv
 - TCR_A_variable_domains.fasta
 - TCR_B_variable_domains.fasta
 - input_rejected.csv
 - input_sequence_manifest.csv
 - pairwise_search_sequence_manifest.csv
 - run_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation checklist

- A CDR sequence comes from IMGT-numbered positions, not raw offsets.
- Use `ANARCI_chain_label_mismatches.csv` to audit TCR3d header labels.
- Use `anarci_bitscore` only as HMM support for an A/B domain.
- Use `blast_bitscore`, identity, coverage, and E-value to describe pairwise similarity.
- A higher BLAST bit score means a stronger local pairwise alignment under this scoring system; it does not by itself prove the same antigen specificity.
